# Chapitre 5 — Nettoyage des données

**Durée estimée : 10-12 heures**

---

## Objectifs d'apprentissage

À la fin de ce chapitre, vous serez capable de :

1. **Appliquer** différentes stratégies de traitement des valeurs manquantes (suppression, imputation)
2. **Identifier et supprimer** les doublons en préservant les informations pertinentes
3. **Traiter** les valeurs aberrantes selon le contexte métier
4. **Nettoyer** les types de données (dates, numériques, texte) pour les rendre exploitables

---

## 5.4 Nettoyage des types de données

### Conversion des dates

In [ ]:
import pandas as pd

# Données avec différents formats de dates
df_dates = pd.DataFrame({
    'date_iso': ['2024-01-15', '2024-02-20', '2024-03-25'],
    'date_fr': ['15/01/2024', '20/02/2024', '25/03/2024'],
    'date_us': ['01/15/2024', '02/20/2024', '03/25/2024'],
    'date_erreur': ['15/01/2024', '31/02/2024', '25/03/2024']  # 31 février n'existe pas
})

print("Types avant conversion :")
print(df_dates.dtypes)

# Quand tu crées ton DataFrame, la colonne date_fr contient du texte (type object ou string en Python).
# Pour toi, c'est une date.
# Pour l'ordinateur, c'est juste une suite de caractères, comme "Bonjour".

Types avant conversion :
date_iso       object
date_fr        object
date_us        object
date_erreur    object
dtype: object


In [12]:
# Conversion basique (format ISO)
df_dates['date_iso_clean'] = pd.to_datetime(df_dates['date_iso'])
print("Format ISO converti :")
print(df_dates['date_iso_clean'])
print(df_dates.dtypes)

Format ISO converti :
0   2024-01-15
1   2024-02-20
2   2024-03-25
Name: date_iso_clean, dtype: datetime64[ns]
date_iso                     object
date_fr                      object
date_us                      object
date_erreur                  object
date_iso_clean       datetime64[ns]
date_fr_clean        datetime64[ns]
date_erreur_clean    datetime64[ns]
dtype: object


In [ ]:
# Avec format spécifique (français)
df_dates['date_fr_clean'] = pd.to_datetime(df_dates['date_fr'], format='%d/%m/%Y')
print("\nFormat FR converti :")
print(df_dates['date_fr_clean'])

# Le paramètre format='%d/%m/%Y' ne sert pas à changer l'affichage final, il sert de "décodeur". 
# Tu dis à Python : "Regarde ce texte, le premier chiffre est le jour, puis il y a un slash, puis le mois...".
# une fois que Python a converti ton texte en un véritable objet datetime, 
# il l'enregistre dans un format standardisé (ISO 8601) : YYYY-MM-DD.


Format FR converti :
0   2024-01-15
1   2024-02-20
2   2024-03-25
Name: date_fr_clean, dtype: datetime64[ns]


In [ ]:
# Gestion des erreurs (31 février → NaT)
df_dates['date_erreur_clean'] = pd.to_datetime(df_dates['date_erreur'], format='%d/%m/%Y', errors='coerce')
# 'coerce' remplace les erreurs par NaT (Not a Time)
print("\nFormat avec erreur (31/02 → NaT) :")
print(df_dates['date_erreur_clean'])


Format avec erreur (31/02 → NaT) :
0   2024-01-15
1          NaT
2   2024-03-25
Name: date_erreur_clean, dtype: datetime64[ns]


Le paramètre format : C'est le "Mode d'emploi" de lecture. 

Tu dis à Python : « Attention, dans ce texte que je te donne, les chiffres sont rangés dans cet ordre précis : Jour / Mois / Année ». C'est l'étape de compréhension.

Le stockage (le résultat) : Une fois que Python a compris quel chiffre correspond à quoi, il ne garde pas le texte original. 

Il transforme l'information en un objet binaire datetime64. Et cet objet, par convention informatique internationale, s'affiche toujours par défaut sous la forme YYYY-MM-DD.

In [ ]:
# On transforme l'objet date en texte joli (pour un rapport)
df_dates['date_jolie'] = df_dates['date_fr_clean'].dt.strftime('%d %B %Y')
print( "\nDate au format joli :")
print(df_dates['date_jolie'])


Date au format joli :
0     15 January 2024
1    20 February 2024
2       25 March 2024
Name: date_jolie, dtype: object


### Extraction de composantes temporelles

In [5]:
# Créer une colonne date
df_temps = pd.DataFrame({
    'date': pd.to_datetime(['2024-01-15', '2024-06-20', '2024-12-25'])
})

# Extraire des composantes temporelles
df_temps['annee'] = df_temps['date'].dt.year
df_temps['mois'] = df_temps['date'].dt.month
df_temps['jour'] = df_temps['date'].dt.day
df_temps['jour_semaine'] = df_temps['date'].dt.dayofweek  # 0=lundi
df_temps['trimestre'] = df_temps['date'].dt.quarter
df_temps['semaine'] = df_temps['date'].dt.isocalendar().week

print("Composantes extraites :")
print(df_temps)

Composantes extraites :
        date  annee  mois  jour  jour_semaine  trimestre  semaine
0 2024-01-15   2024     1    15             0          1        3
1 2024-06-20   2024     6    20             3          2       25
2 2024-12-25   2024    12    25             2          4       52


### Conversion des numériques

In [7]:
# Données avec formats variés
df_num = pd.DataFrame({
    'prix_fr': ['1 234,56 €', '987,00 €', '45,99 €'],
    'taux': ['5,5%', '20%', '10%'],
    'quantite': ['100', '50', 'vingt']  # 'vingt' n'est pas convertible
})

print("Types avant conversion :")
print(df_num.dtypes)

Types avant conversion :
prix_fr     object
taux        object
quantite    object
dtype: object


In [9]:
# Gestion des formats français (virgule décimale, espaces, €)
df_num['prix_clean'] = df_num['prix_fr'].str.replace(' ', '').str.replace('€', '').str.replace(',', '.')
# str.strip() enlève les espaces avant et après que pour les chaines de caractères
df_num['prix_clean'] = pd.to_numeric(df_num['prix_clean'])
print("Prix nettoyés :")
print(df_num['prix_clean'])

Prix nettoyés :
0    1234.56
1     987.00
2      45.99
Name: prix_clean, dtype: float64


In [10]:
# Pourcentages en décimales
df_num['taux_clean'] = df_num['taux'].str.replace(',', '.').str.replace('%', '')
df_num['taux_clean'] = pd.to_numeric(df_num['taux_clean']) / 100
# to_numeric convertit en float

print("\nTaux en décimales :")
print(df_num['taux_clean'])


Taux en décimales :
0    0.055
1    0.200
2    0.100
Name: taux_clean, dtype: float64


In [11]:
# Gestion des erreurs de conversion
df_num['quantite_clean'] = pd.to_numeric(df_num['quantite'], errors='coerce')
# to_numeric avec errors='coerce' remplace les erreurs par NaN

print("\nQuantités (erreurs → NaN) :")
print(df_num['quantite_clean'])


Quantités (erreurs → NaN) :
0    100.0
1     50.0
2      NaN
Name: quantite_clean, dtype: float64


### ✍️ Exercice 5.5 : Conversion de types (15 min)

In [ ]:
import pandas as pd

# Données avec formats variés
df_ex5 = pd.DataFrame({
    'date_fr': ['15/01/2024', '28/02/2024', '31/04/2024', '10/03/2024'],  # 31 avril n'existe pas
    'prix_fr': ['1 234,56 €', '987,00 €', '45,99 €', '2 500,00 €'],
    'taux': ['5,5%', '20%', '10%', '7,5%'],
    'quantite': ['100', '50', 'vingt', '75']
})

print("Avant conversion :")
print(df_ex5.dtypes)

In [ ]:
# 1. Convertir les dates (gérer l'erreur du 31 avril)
df_ex5['date_clean'] = pd.to_datetime(df_ex5['date_fr'], format='%d/%m/%Y', errors='coerce')

# 2. Convertir les prix
df_ex5['prix_clean'] = df_ex5['prix_fr'].str.replace(' ', '').str.replace('€', '').str.replace(',', '.')
df_ex5['prix_clean'] = pd.to_numeric(df_ex5['prix_clean'])

# 3. Convertir les taux en décimales
df_ex5['taux_clean'] = df_ex5['taux'].str.replace(',', '.').str.replace('%', '')
df_ex5['taux_clean'] = pd.to_numeric(df_ex5['taux_clean']) / 100

# 4. Convertir les quantités (gérer 'vingt')
df_ex5['quantite_clean'] = pd.to_numeric(df_ex5['quantite'], errors='coerce')

print("\nAprès conversion :")
print(df_ex5[['date_clean', 'prix_clean', 'taux_clean', 'quantite_clean']])

print("\nTypes après conversion :")
print(df_ex5[['date_clean', 'prix_clean', 'taux_clean', 'quantite_clean']].dtypes)